Read in all files 

In [2]:
import pandas as pd
from pathlib import Path
import numpy as np


#read input files 
IO_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Industry\flatfile_eu-ic-io_ind-by-ind_25ed_2023\flatfile_eu-ic-io_ind-by-ind_25ed_2023.csv") #input output tables for industry
IO_matrix = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Industry\matrix_eu-ic-io_ind-by-ind_25ed_2023.csv")
GHG_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Emissions\env_ac_ghgfp_2023_cleaned.csv") #Emissions 
IMPORT_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Product\EU_RES_IMPORT_FILE.csv") #EU imported renewable energy sources
region_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\regions.csv") #csv for mapping regions to correct names
buildout_linear = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\Material\Buildout_spores\buildout_linear.csv") #Linear buildout till 2050
buildout_gompertz = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\Material\Buildout_spores\buildout_gompertz.csv") #Gompertz growth buildout till 2050
buildout_logistics = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\Material\Buildout_spores\buildout_logistic.csv") #Logistics growth buildout till 2050

df_IO = pd.read_csv(IO_file, sep=",")
df_matrix = pd.read_csv(IO_matrix)
df_GHG = pd.read_csv(GHG_file, sep=";", encoding="utf-8-sig", low_memory=False)
df_IMPORT = pd.read_csv(IMPORT_file, sep=",", encoding="utf-8-sig", low_memory=False)
df_regions = pd.read_csv(region_file, sep=";", encoding="utf-8-sig", low_memory=False)
df_linear_buildout = pd.read_csv(buildout_linear)
df_gompertz_buildout = pd.read_csv(buildout_gompertz)
df_logistics_buildout = pd.read_csv(buildout_logistics)

In [3]:
# Give countries the right names & only take into account the 46 individual countries (code other countries as Rest Of World)
# Input/Output file
io_to_iso3 = (
    df_regions
    .dropna(subset=["IO", "ISO_3"])
    .set_index("IO")["ISO_3"]
    .to_dict()
)

for column in ["refArea", "counterpartArea"]:
    df_IO[column] = df_IO[column].map(io_to_iso3).fillna("ROW")
    
    
# Matrix version

df_matrix = df_matrix.set_index("rowLabels")
df_matrix.index = df_matrix.index.astype(str).str.strip()
df_matrix.columns = df_matrix.columns.astype(str).str.strip()
df_matrix = df_matrix.apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0)

def map_io_label(label, mapping):
    label = str(label).strip()
    country, sector = str(label).split("_", 1)
    country_iso3 = mapping.get(country, "ROW")
    return f"{country_iso3}_{sector}"


df_matrix.index = [
    map_io_label(label, io_to_iso3)
    for label in df_matrix.index
]

df_matrix.columns = [
    map_io_label(label, io_to_iso3)
    for label in df_matrix.columns
]
    
    
# GHG file    
GHG_to_iso3 = (
    df_regions
    .dropna(subset=["GHG", "ISO_3"])
    .set_index("GHG")["ISO_3"]
    .to_dict()
)

for column in ["c_dest", "c_orig"]:
    df_GHG[column] = df_GHG[column].map(GHG_to_iso3).fillna("ROW")
    
# Import file
IMPORT_to_iso3 = (
    df_regions
    .dropna(subset=["Import", "ISO_3"])
    .set_index("Import")["ISO_3"]
    .to_dict()
)

for column in ["reporter", "partner"]:
    df_IMPORT[column] = df_IMPORT[column].map(IMPORT_to_iso3).fillna("ROW")
    
# Remove aggregates
io_aggregates_refArea = [
    "W2"
]

ghg_aggregates_c_dest = [
    "EXTRA-EU-27",
    "EU-27"
]

ghg_aggregates_c_orig = [
    "EXTRA-EU-27",
    "EU-27",
    "ALL"
]

import_aggregates_partner = [
    "EXTRA-EU-27",
    "EXTRA-EA-21",
    "EXTRA-EA",
    "INT-EA"
]

import_product = [
    "Wood pellets"
]

# Remove from IO file
df_IO = df_IO.loc[
    ~df_IO["refArea"].isin(io_aggregates_refArea)
].copy()


# Remove from GHG file
df_GHG = df_GHG.loc[
    (~df_GHG["c_dest"].isin(ghg_aggregates_c_dest)) &
    (~df_GHG["c_orig"].isin(ghg_aggregates_c_orig))
].copy()


# Remove from import file
df_IMPORT = df_IMPORT.loc[
    (~df_IMPORT["partner"].isin(import_aggregates_partner))&
    (~df_IMPORT["product"].isin(import_product))
].copy()
    
    
# Group products into tech sources    
products_to_techs = {
    #"Wood pellets" : "bio fuel",
    "Photovoltaic cells assembled in modules or made up into panels" : "solar",
    "Photovoltaic cells not assembled in modules or made up into panels" : "solar",
    "Static converters" : "solar",
    "Electric accumulators (excl. spent and lead-acid, nickel-cadmium, nickel-iron, nickel-metal hydride and lithium-ion accumulators)(2022-2500);Electric accumulators (excl. spent and lead-acid, nickel-cadmium, nickel-iron, nickel-metal hydride and lithium-ion accumulators)(2012-2021)" : "batteries",
    "Lithium-ion accumulators (excl. spent)" : "batteries",
    "Generating sets, wind-powered(2002-2500);Generating sets, wind-powered(1996-2001)" : "wind"
}

df_IMPORT["product"] = df_IMPORT["product"].replace(products_to_techs)

In [4]:
# Add up all rows with same values
# Input/Output file
df_IO["obsValue"] = pd.to_numeric(
    df_IO["obsValue"],
    errors="coerce"
).fillna(0)

df_IO = (
    df_IO
    .groupby(
        ["refArea", "rowIi", "counterpartArea", "colIi"],
        as_index=False,
        dropna=False
    )["obsValue"]
    .sum()
)



df_IO["icioiRow"] = (
    df_IO["refArea"].astype(str)
    + "_"
    + df_IO["rowIi"].astype(str)
)

df_IO["icioiCol"] = (
    df_IO["counterpartArea"].astype(str)
    + "_"
    + df_IO["colIi"].astype(str)
)

df_IO = df_IO[
    [
        "icioiRow",
        "icioiCol",
        "refArea",
        "rowIi",
        "counterpartArea",
        "colIi",
        "obsValue"
    ]
]


# Matrix file
df_matrix = df_matrix.groupby(df_matrix.index).sum()
df_matrix = df_matrix.T.groupby(df_matrix.columns).sum().T


# GHG file
df_GHG["OBS_VALUE"] = (
    df_GHG["OBS_VALUE"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df_GHG["OBS_VALUE"] = pd.to_numeric(
    df_GHG["OBS_VALUE"],
    errors="coerce"
).fillna(0)

df_GHG = (
    df_GHG
    .groupby(
        ["c_dest", "na_item", "c_orig", "nace_r2", "nace_r2_code"],
        as_index=False,
        dropna=False
    )["OBS_VALUE"]
    .sum()
)

# Import file
df_IMPORT["OBS_VALUE"] = (
    df_IMPORT["OBS_VALUE"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df_IMPORT["OBS_VALUE"] = pd.to_numeric(
    df_IMPORT["OBS_VALUE"],
    errors="coerce"
).fillna(0)

df_IMPORT = (
    df_IMPORT
    .groupby(
        ["reporter", "partner", "product"],
        as_index=False,
        dropna=False
    )["OBS_VALUE"]
    .sum()
)

EE-MRIO analysis

In [5]:
# Some model parameters
EU_countries = [
    "AUT",  # Austria
    "BEL",  # Belgium
    "BGR",  # Bulgaria
    "CYP",  # Cyprus
    "CZE",  # Czechia
    "DEU",  # Germany
    "DNK",  # Denmark
    "ESP",  # Spain
    "EST",  # Estonia
    "FIN",  # Finland
    "FRA",  # France
    "GRC",  # Greece
    "HRV",  # Croatia
    "HUN",  # Hungary
    "IRL",  # Ireland
    "ITA",  # Italy
    "LVA",  # Latvia
    "LTU",  # Lithuania
    "LUX",  # Luxembourg
    "NLD",  # Netherlands
    "POL",  # Poland
    "PRT",  # Portugal
    "ROU",  # Romania
    "SVK",  # Slovakia
    "SVN",  # Slovenia
    "SWE",  # Sweden
] # Malta not included

dollar_to_euro_2023 = 0.9242

capacity_price = {"solar": 261e6 * dollar_to_euro_2023, "offshore wind": 700e6 * dollar_to_euro_2023, "onshore wind": 700e6 * dollar_to_euro_2023, "batteries": 260e6 * dollar_to_euro_2023} #Euro/(TW/TWh)
construction_price = {"solar": (758e6 - 261e6) * dollar_to_euro_2023, "offshore wind": (2800e6 - 700e6) * dollar_to_euro_2023, "onshore wind": (1160e6 - 700e6) * dollar_to_euro_2023, "batteries": (273e6 - 260e6) * dollar_to_euro_2023} #Euro/(TW/TWh)

final_demand = ["P3_S13", "P3_S14", "P3_S15", "P51G", "P5M"] #Nace codes for final demand

value_added = ["D1", "D21X31", "D29X39", "B2A3G", "OP_RES", "OP_NRES"] #Nace codes for value added

res_code = {"batteries" : "C27", "solar" : "C27", "wind": "C28", "construction" : "F"} #Nace codes for technologies (techs)

In [6]:
# Calculate leontief inverse values to determine upstream emissions

# Differentiate between Intermediate use bock & Final use block 
def get_sector(label):
    return str(label).split("_", 1)[1]


row_sectors = pd.Series(df_matrix.index, index=df_matrix.index).apply(get_sector)
col_sectors = pd.Series(df_matrix.columns, index=df_matrix.columns).apply(get_sector)

production_rows = row_sectors.loc[
    ~row_sectors.isin(value_added)
].index

sector_columns = col_sectors.loc[
    ~col_sectors.isin(final_demand)
].index

final_demand_columns = col_sectors.loc[
    col_sectors.isin(final_demand)
].index


# Build intermediate transaction matrix Z
Z = df_matrix.loc[
    production_rows,
    sector_columns
].copy()

# Make square matrix
all_sectors = sorted(set(Z.index) | set(Z.columns))

Z = Z.reindex(
    index=all_sectors,
    columns=all_sectors,
    fill_value=0
)

# Build final demand matrix Y
Y = df_matrix.loc[
    production_rows,
    final_demand_columns
].copy()

Y = Y.reindex(
    index=all_sectors,
    fill_value=0
)

# Calculate total output x
x_output = Z.sum(axis=1) + Y.sum(axis=1)

# Keep only sectors with positive output
valid_sectors = x_output[x_output > 0].index

Z = Z.reindex(
    index=valid_sectors,
    columns=valid_sectors,
    fill_value=0
)

Y = Y.reindex(
    index=valid_sectors,
    fill_value=0
)

x_output = x_output.reindex(valid_sectors)

# Calculate technical coefficient matrix A
A = Z.divide(x_output, axis=1).fillna(0)

# Calculate Leontief inverse
I = np.eye(len(A))

L = pd.DataFrame(
    np.linalg.solve(I - A.values, I),
    index=A.index,
    columns=A.columns
)

In [7]:
# Create flat dataframe for L: required_input, country_dest, sector_dest, country_orig, sector_orig
def split_sector_id(sector_id):
    country, sector = str(sector_id).split("_", 1)
    return country, sector

# Select only demand columns whose sector is in res_codes (used in analysis)
res_demand_columns = [
    col for col in L.columns
    if split_sector_id(col)[1] in set(res_code.values())
]

L_res = L.loc[:, res_demand_columns].copy()

df_L_res_flat = (
    L_res
    .stack()
    .reset_index()
)

df_L_res_flat.columns = [
    "required_sector_id",
    "demand_sector_id",
    "required_input"
]

df_L_res_flat[["country_dest", "sector_dest"]] = (
    df_L_res_flat["demand_sector_id"]
    .str.split("_", n=1, expand=True)
)
df_L_res_flat[["country_orig", "sector_orig"]] = (
    df_L_res_flat["required_sector_id"]
    .str.split("_", n=1, expand=True)
)

del df_L_res_flat["required_sector_id"]
del df_L_res_flat["demand_sector_id"]

In [8]:
# Dataframe with total emissions caused by each sector in each country: country_orig, sector, total_emissions
df_emissions = (
    df_GHG
    .loc[df_GHG["na_item"] == "Total"]
    .groupby(["c_orig", "nace_r2_code"], as_index=False)["OBS_VALUE"]
    .sum()
    .rename(columns={
            "c_orig": "country_orig",
            "nace_r2_code": "sector",
            "OBS_VALUE": "total_emissions"
        })
    
)

# Dataframe with final outputs by each sector in each country: country_orig, sector, total_output
df_total_output = (
    df_IO
    .groupby(["refArea", "rowIi"], as_index=False)["obsValue"]
    .sum()
    .rename(columns={
        "refArea": "country_orig",
        "rowIi": "sector",
        "obsValue": "total_output"
    })
)

# Dataframe with  direct emissions per euro for each sector in each country: country_orig, sector, total_emissions, total_output, direct_emissions_per_euro
df_direct_emissions_per_euro = df_emissions.merge(
    df_total_output[
        ["country_orig", "sector", "total_output"]
    ],
    on=["country_orig", "sector"],
    how="left"
)

df_direct_emissions_per_euro = df_direct_emissions_per_euro.rename(columns={
    "country_orig": "country"
})


df_direct_emissions_per_euro = df_direct_emissions_per_euro.dropna(
    subset=["total_output"]
).copy()

df_direct_emissions_per_euro = df_direct_emissions_per_euro.loc[
    df_direct_emissions_per_euro["total_output"] > 0
].copy()

df_direct_emissions_per_euro["direct_emissions_per_euro"] = df_direct_emissions_per_euro["total_emissions"] / df_direct_emissions_per_euro["total_output"]

In [9]:
# Dataframe with required emissions for 1 euro output from country_dest,sector_dest (upstream emissions)
# DF: required_input, country_dest, sector_dest, country_orig, sector_orig, direct_emissions_per_euro, upstream_emission_per_euro (direct_emissions_per_euro is for specific combination country_orig,sector_orig)
df_upstream_emissions_per_euro = df_L_res_flat.merge(
    df_direct_emissions_per_euro[
        ["country", "sector", "direct_emissions_per_euro"]
    ],
    left_on=["country_orig", "sector_orig"],
    right_on=["country", "sector"],
    how="left"
)


df_upstream_emissions_per_euro["upstream_emissions_per_euro"] = (
    df_upstream_emissions_per_euro["required_input"] *
    df_upstream_emissions_per_euro["direct_emissions_per_euro"]
)


# Dataframe with 
# Dataframe with total upstream emissions per euro for each country for each sector in the analysis
df_total_upstream_emissions_per_euro = (
    df_upstream_emissions_per_euro
    .groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
    .rename(columns={
        "upstream_emissions_per_euro": "total_upstream_emissions_per_euro",
        "country_dest": "country",
        "sector_dest": "sector"
    }))

In [10]:
# Determine where upstream emissions come from (domestic, EU, non-EU)

df_upstream_emissions_per_euro_self = df_upstream_emissions_per_euro.loc[
    (df_upstream_emissions_per_euro["country_dest"] == df_upstream_emissions_per_euro["country_orig"])&
    (df_upstream_emissions_per_euro["sector_dest"].isin(res_code.values()))
].copy()


df_total_upstream_emissions_per_euro_self = (
    df_upstream_emissions_per_euro_self
    .groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
    .rename(columns={
        "upstream_emissions_per_euro": "total_upstream_emissions_per_euro_self",
        "country_dest": "country",
        "sector_dest": "sector"
    }))



df_upstream_emissions_per_euro_EU = df_upstream_emissions_per_euro.loc[
    df_upstream_emissions_per_euro["country_orig"].isin(EU_countries)&
    (df_upstream_emissions_per_euro["sector_dest"].isin(res_code.values()))
].copy()

df_upstream_emissions_per_euro_EU = df_upstream_emissions_per_euro_EU[df_upstream_emissions_per_euro_EU["country_dest"] != df_upstream_emissions_per_euro_EU["country_orig"]]

df_total_upstream_emissions_per_euro_EU = (
    df_upstream_emissions_per_euro_EU
    .groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
    .rename(columns={
        "upstream_emissions_per_euro": "total_upstream_emissions_per_euro_EU",
        "country_dest": "country",
        "sector_dest": "sector"
    }))

df_upstream_emissions_per_euro_non_EU = df_upstream_emissions_per_euro.loc[
    ~df_upstream_emissions_per_euro["country_orig"].isin(EU_countries)&
    (df_upstream_emissions_per_euro["sector_dest"].isin(res_code.values()))
].copy()

df_upstream_emissions_per_euro_non_EU = df_upstream_emissions_per_euro_non_EU[df_upstream_emissions_per_euro_non_EU["country_dest"] != df_upstream_emissions_per_euro_non_EU["country_orig"]]

df_total_upstream_emissions_per_euro_non_EU = (
    df_upstream_emissions_per_euro_non_EU
    .groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
    .rename(columns={
        "upstream_emissions_per_euro": "total_upstream_emissions_per_euro_non_EU",
        "country_dest": "country",
        "sector_dest": "sector"
    }))



In [11]:
# Calculate the share of emissions caused domesticly, inside EU, outside EU

df_upstream_emission_shares = (
    df_total_upstream_emissions_per_euro
    .merge(
        df_total_upstream_emissions_per_euro_self,
        on=["country", "sector"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_total_upstream_emissions_per_euro_EU,
        on=["country", "sector"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_total_upstream_emissions_per_euro_non_EU,
        on=["country", "sector"],
        how="left",
        validate="one_to_one"
    )
)

split_columns = [
    "total_upstream_emissions_per_euro_self",
    "total_upstream_emissions_per_euro_EU",
    "total_upstream_emissions_per_euro_non_EU"
]

df_upstream_emission_shares[split_columns] = (
    df_upstream_emission_shares[split_columns].fillna(0)
)

df_upstream_emission_shares["share_self"] = np.where(
    df_upstream_emission_shares["total_upstream_emissions_per_euro"] > 0,
    df_upstream_emission_shares["total_upstream_emissions_per_euro_self"] /
    df_upstream_emission_shares["total_upstream_emissions_per_euro"],
    0
)

df_upstream_emission_shares["share_EU"] = np.where(
    df_upstream_emission_shares["total_upstream_emissions_per_euro"] > 0,
    df_upstream_emission_shares["total_upstream_emissions_per_euro_EU"] /
    df_upstream_emission_shares["total_upstream_emissions_per_euro"],
    0
)

df_upstream_emission_shares["share_non_EU"] = np.where(
    df_upstream_emission_shares["total_upstream_emissions_per_euro"] > 0,
    df_upstream_emission_shares["total_upstream_emissions_per_euro_non_EU"] /
    df_upstream_emission_shares["total_upstream_emissions_per_euro"],
    0
)

In [12]:
# Dataframe with information on import share for each country for each tech: country_dest, country_orig, tech, import, total import, import_share
df_import_shares = (
    df_IMPORT
    .groupby(["reporter", "partner", "product"], as_index=False)["OBS_VALUE"]
    .sum()
    .rename(columns={
        "reporter": "country_dest",
        "partner": "country_orig",
        "product": "tech",
        "OBS_VALUE": "import"
    })
)

# Total import for each country for each tech
df_import_shares["total_import"] = (
df_import_shares
.groupby(["country_dest", "tech"])["import"]
.transform("sum")
)

#Share of origin country in total import
df_import_shares["import_share"] = np.where(
df_import_shares["total_import"] > 0,
df_import_shares["import"] / df_import_shares["total_import"],
np.nan
)

In [13]:
# Create dataframe with finel total upstream emissions shares based on import data

df_import_shares["sector"] = (
    df_import_shares["tech"].map(res_code)
)

# Exclude construction part because this is assumed to be domestic
df_upstream_emission_shares_no_F = df_upstream_emission_shares.loc[
    df_upstream_emission_shares["sector"] != "F"
].copy()

df_upstream_location_shares = df_import_shares.merge(
    df_upstream_emission_shares_no_F[
        [
            "country",
            "sector",
            "share_self",
            "share_EU",
            "share_non_EU"
        ]
    ],
    left_on=["country_orig", "sector"],
    right_on=["country", "sector"],
    how="left",
    validate="many_to_one"
)

df_upstream_location_shares["weighted_share_self"] = (
    df_upstream_location_shares["import_share"] *
    df_upstream_location_shares["share_self"]
)

df_upstream_location_shares["weighted_share_EU"] = (
    df_upstream_location_shares["import_share"] *
    df_upstream_location_shares["share_EU"]
)

df_upstream_location_shares["weighted_share_non_EU"] = (
    df_upstream_location_shares["import_share"] *
    df_upstream_location_shares["share_non_EU"]
)

df_upstream_location_shares = df_upstream_location_shares[
    [
        "country_dest",
        "country_orig",
        "tech",
        "sector",
        "import",
        "total_import",
        "import_share",
        "weighted_share_self",
        "weighted_share_EU",
        "weighted_share_non_EU"
    ]
].copy()

df_total_upstream_location_shares = (
    df_upstream_location_shares
    .groupby(["country_dest", "tech"], as_index=False)[
        ["weighted_share_self", "weighted_share_EU", "weighted_share_non_EU"]
    ]
    .sum()
)

# determine total upstream emission share for construction

df_total_upstream_location_shares_F = df_upstream_emission_shares.loc[
    df_upstream_emission_shares["sector"] == "F"
]

In [14]:
# Create dataframe with final upstream emissions per euro for each country & tech (based on import shares)
# Dataframe with weighted upstream emissions: country_dest, country_orig, tech, import, total import, import_share, sector, upstream_emissions_per_euro, weighted_upstream_emissions_per_euro (weighted_upstream_emissions_per_euro = upstream_emissions_per_euro * import_share)
df_weighted_upstream_emissions_per_euro = df_import_shares.copy()

df_weighted_upstream_emissions_per_euro["sector"] = (
    df_weighted_upstream_emissions_per_euro["tech"].map(res_code)
)

df_upstream_factor = df_total_upstream_emissions_per_euro.rename(columns={
    "country": "country_orig",
    "sector_dest": "sector",
    "total_upstream_emissions_per_euro": "upstream_emissions_per_euro"
})

df_weighted_upstream_emissions_per_euro = df_weighted_upstream_emissions_per_euro.merge(
    df_upstream_factor[
        ["country_orig", "sector", "upstream_emissions_per_euro"]
    ],
    on=["country_orig", "sector"],
    how="left",
    validate="many_to_one"
)

df_weighted_upstream_emissions_per_euro["weighted_upstream_emissions_per_euro"] = (
df_weighted_upstream_emissions_per_euro["import_share"] *
df_weighted_upstream_emissions_per_euro["upstream_emissions_per_euro"]
)

# Dataframe with final upstream emissions per euro for each country for each tech: country, tech, total_upstream_emissions_per_euro
df_final_upstream_emissions_per_euro = (
df_weighted_upstream_emissions_per_euro
.groupby(["country_dest", "tech"], as_index=False)["weighted_upstream_emissions_per_euro"]
.sum()
.rename(columns={
"country_dest": "country",
"weighted_upstream_emissions_per_euro": "total_upstream_emissions_per_euro"})
)

df_upstream_emissions_per_euro_F = df_upstream_emissions_per_euro.loc[
    df_upstream_emissions_per_euro["sector_dest"] == "F"
].copy()

df_final_upstream_emissions_per_euro_F = (
df_upstream_emissions_per_euro_F
.groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
.sum()
.rename(columns={
"country_dest": "country",
"sector_dest": "sector",
"upstream_emissions_per_euro": "total_upstream_emissions_per_euro"})
)

In [15]:
# Dataframes with the total annual investment in euro for each EU country, for each tech (from 2026 to 2050)

# Annual investment with Gompertz growth model
df_annual_investment_gompertz = (
df_gompertz_buildout
.groupby(["spore", "country", "tech", "year"], as_index=False)["new_capacity_required_GW"]
.sum()
)

df_annual_investment_gompertz["capacity_price"] = (
df_annual_investment_gompertz["tech"].map(capacity_price)
)

df_annual_investment_gompertz["construction_price"] = (
df_annual_investment_gompertz["tech"].map(construction_price)
)


df_annual_investment_gompertz["investment"] = np.where(
df_annual_investment_gompertz["new_capacity_required_GW"] > 0,
df_annual_investment_gompertz["new_capacity_required_GW"] * df_annual_investment_gompertz["capacity_price"],
np.nan
)

df_annual_investment_gompertz["construction_cost"] = np.where(
df_annual_investment_gompertz["new_capacity_required_GW"] > 0,
df_annual_investment_gompertz["new_capacity_required_GW"] * df_annual_investment_gompertz["construction_price"],
np.nan
)


# Annual investment with Logistics growth model
df_annual_investment_logistics = (
df_logistics_buildout
.groupby(["spore", "country", "tech", "year"], as_index=False)["new_capacity_required_GW"]
.sum()
)

df_annual_investment_logistics["capacity_price"] = (
df_annual_investment_logistics["tech"].map(capacity_price)
)

df_annual_investment_logistics["construction_price"] = (
df_annual_investment_logistics["tech"].map(construction_price)
)

df_annual_investment_logistics["investment"] = np.where(
df_annual_investment_logistics["new_capacity_required_GW"] > 0,
df_annual_investment_logistics["new_capacity_required_GW"] * df_annual_investment_logistics["capacity_price"],
np.nan
)

df_annual_investment_logistics["construction_cost"] = np.where(
df_annual_investment_logistics["new_capacity_required_GW"] > 0,
df_annual_investment_logistics["new_capacity_required_GW"] * df_annual_investment_logistics["construction_price"],
np.nan
)



# Annual investment with linear growth model
df_annual_investment_linear = (
df_linear_buildout
.groupby(["spore", "country", "tech", "year"], as_index=False)["new_capacity_required_GW"]
.sum()
)

df_annual_investment_linear["capacity_price"] = (
df_annual_investment_linear["tech"].map(capacity_price)
)

df_annual_investment_linear["construction_price"] = (
df_annual_investment_linear["tech"].map(construction_price)
)

df_annual_investment_linear["investment"] = np.where(
df_annual_investment_linear["new_capacity_required_GW"] > 0,
df_annual_investment_linear["new_capacity_required_GW"] * df_annual_investment_linear["capacity_price"],
np.nan
)

df_annual_investment_linear["construction_cost"] = np.where(
df_annual_investment_linear["new_capacity_required_GW"] > 0,
df_annual_investment_linear["new_capacity_required_GW"] * df_annual_investment_linear["construction_price"],
np.nan
)

In [17]:
# Dataframe with annual imported emissions per country

# Replace "onshore wind" & "offshore wind" with wind (like in final_upstream_emissions_per_euro dataframe)
df_annual_investment_gompertz = df_annual_investment_gompertz.copy()
df_annual_investment_logistics = df_annual_investment_logistics.copy()
df_annual_investment_linear = df_annual_investment_linear.copy()

df_annual_investment_gompertz["tech"] = (
    df_annual_investment_gompertz["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)
df_annual_investment_logistics["tech"] = (
    df_annual_investment_logistics["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)
df_annual_investment_linear["tech"] = (
    df_annual_investment_linear["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)

# Dataframes with imported emissions

# Linear model
annual_imported_emissions_per_country_linear = (
    df_annual_investment_linear
    .merge(
        df_final_upstream_emissions_per_euro.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)

annual_imported_emissions_per_country_linear["investment"] = (
    annual_imported_emissions_per_country_linear["investment"].fillna(0)
)

annual_imported_emissions_per_country_linear["imported_emissions_tonnes"] = (
    (annual_imported_emissions_per_country_linear["investment"] *
    annual_imported_emissions_per_country_linear["total_upstream_emissions_per_sector"])
    /1000
)

annual_imported_emissions_per_country_linear = annual_imported_emissions_per_country_linear[
    [
        "spore",
        "country",
        "tech",
        "year",
        "investment",
        "total_upstream_emissions_per_sector",
        "imported_emissions_tonnes"
    ]
].copy()

# Gompertz model
annual_imported_emissions_per_country_gompertz = (
    df_annual_investment_gompertz
    .merge(
        df_final_upstream_emissions_per_euro.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)

annual_imported_emissions_per_country_gompertz["investment"] = (
    annual_imported_emissions_per_country_gompertz["investment"].fillna(0)
)

annual_imported_emissions_per_country_gompertz["imported_emissions_tonnes"] = (
    (annual_imported_emissions_per_country_gompertz["investment"] *
    annual_imported_emissions_per_country_gompertz["total_upstream_emissions_per_sector"])
    /1000
)

annual_imported_emissions_per_country_gompertz = annual_imported_emissions_per_country_gompertz[
    [
        "spore",
        "country",
        "tech",
        "year",
        "investment",
        "total_upstream_emissions_per_sector",
        "imported_emissions_tonnes"
    ]
].copy()

# Logistic model
annual_imported_emissions_per_country_logistic = (
    df_annual_investment_logistics
    .merge(
        df_final_upstream_emissions_per_euro.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)

annual_imported_emissions_per_country_logistic["investment"] = (
    annual_imported_emissions_per_country_logistic["investment"].fillna(0)
)

annual_imported_emissions_per_country_logistic["imported_emissions_tonnes"] = (
    (annual_imported_emissions_per_country_logistic["investment"] *
    annual_imported_emissions_per_country_logistic["total_upstream_emissions_per_sector"])
    /1000
)

annual_imported_emissions_per_country_logistic = annual_imported_emissions_per_country_logistic[
    [
        "spore",
        "country",
        "tech",
        "year",
        "investment",
        "total_upstream_emissions_per_sector",
        "imported_emissions_tonnes"
    ]
].copy()



In [18]:
# Create dataframe which shows where upstream emissions come from (domestic, EU, non-EU) for all countries, for all technologies

#Linear model

annual_imported_emissions_location_linear = (
    annual_imported_emissions_per_country_linear
    .merge(
        df_total_upstream_location_shares.rename(columns={
            "country_dest": "country"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)



annual_imported_emissions_location_linear["imported_emissions_tonnes_self"] = (
    annual_imported_emissions_location_linear["imported_emissions_tonnes"] *
    annual_imported_emissions_location_linear["weighted_share_self"]
)

annual_imported_emissions_location_linear["imported_emissions_tonnes_EU"] = (
    annual_imported_emissions_location_linear["imported_emissions_tonnes"] *
    annual_imported_emissions_location_linear["weighted_share_EU"]
)

annual_imported_emissions_location_linear["imported_emissions_tonnes_non_EU"] = (
    annual_imported_emissions_location_linear["imported_emissions_tonnes"] *
    annual_imported_emissions_location_linear["weighted_share_non_EU"]
)

annual_imported_emissions_location_linear["imported_emissions_tonnes_total"] = (
    annual_imported_emissions_location_linear["imported_emissions_tonnes"]
)


annual_imported_emissions_location_linear = annual_imported_emissions_location_linear[
    [
        "spore",
        "country",
        "tech",
        "year",
        "imported_emissions_tonnes_self",
        "imported_emissions_tonnes_EU",
        "imported_emissions_tonnes_non_EU",
        "imported_emissions_tonnes_total"
    ]
].copy()

# Logistic model

annual_imported_emissions_location_logistic = (
    annual_imported_emissions_per_country_logistic
    .merge(
        df_total_upstream_location_shares.rename(columns={
            "country_dest": "country"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)


annual_imported_emissions_location_logistic["imported_emissions_tonnes_self"] = (
    annual_imported_emissions_location_logistic["imported_emissions_tonnes"] *
    annual_imported_emissions_location_logistic["weighted_share_self"]
)

annual_imported_emissions_location_logistic["imported_emissions_tonnes_EU"] = (
    annual_imported_emissions_location_logistic["imported_emissions_tonnes"] *
    annual_imported_emissions_location_logistic["weighted_share_EU"]
)

annual_imported_emissions_location_logistic["imported_emissions_tonnes_non_EU"] = (
    annual_imported_emissions_location_logistic["imported_emissions_tonnes"] *
    annual_imported_emissions_location_logistic["weighted_share_non_EU"]
)

annual_imported_emissions_location_logistic["imported_emissions_tonnes_total"] = (
    annual_imported_emissions_location_logistic["imported_emissions_tonnes"]
)


annual_imported_emissions_location_logistic = annual_imported_emissions_location_logistic[
    [
        "spore",
        "country",
        "tech",
        "year",
        "imported_emissions_tonnes_self",
        "imported_emissions_tonnes_EU",
        "imported_emissions_tonnes_non_EU",
        "imported_emissions_tonnes_total"
    ]
].copy()


# Gompertz model

annual_imported_emissions_location_gompertz = (
    annual_imported_emissions_per_country_gompertz
    .merge(
        df_total_upstream_location_shares.rename(columns={
            "country_dest": "country"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)



annual_imported_emissions_location_gompertz["imported_emissions_tonnes_self"] = (
    annual_imported_emissions_location_gompertz["imported_emissions_tonnes"] *
    annual_imported_emissions_location_gompertz["weighted_share_self"]
)

annual_imported_emissions_location_gompertz["imported_emissions_tonnes_EU"] = (
    annual_imported_emissions_location_gompertz["imported_emissions_tonnes"] *
    annual_imported_emissions_location_gompertz["weighted_share_EU"]
)

annual_imported_emissions_location_gompertz["imported_emissions_tonnes_non_EU"] = (
    annual_imported_emissions_location_gompertz["imported_emissions_tonnes"] *
    annual_imported_emissions_location_gompertz["weighted_share_non_EU"]
)

annual_imported_emissions_location_gompertz["imported_emissions_tonnes_total"] = (
    annual_imported_emissions_location_gompertz["imported_emissions_tonnes"]
)


annual_imported_emissions_location_gompertz = annual_imported_emissions_location_gompertz[
    [
        "spore",
        "country",
        "tech",
        "year",
        "imported_emissions_tonnes_self",
        "imported_emissions_tonnes_EU",
        "imported_emissions_tonnes_non_EU",
        "imported_emissions_tonnes_total"
    ]
].copy()

In [19]:
# Dataframe which calculates share of emissions from each sector

df_sector_total = (
    df_upstream_emissions_per_euro
    .groupby(["country_dest", "sector_dest", "sector_orig"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
)

df_sector_shares = df_sector_total.merge(
    df_total_upstream_emissions_per_euro[
        ["country", "sector", "total_upstream_emissions_per_euro"]
    ],
    left_on=["country_dest", "sector_dest"],
    right_on=["country", "sector"],
    how="left"
)
df_sector_shares["sector_share"] = df_sector_shares["upstream_emissions_per_euro"]/df_sector_shares["total_upstream_emissions_per_euro"]

df_sector_shares = df_sector_shares[
    [
        "country_dest",
        "sector_dest",
        "sector_orig",
        "sector_share"
    ]
].copy()

In [20]:
# Dataframe with import result totals from linear model

# Total imported emissions in the EU every year for every tech
df_total_imported_emissions_EU_linear = (
    annual_imported_emissions_location_linear
    .groupby(["spore", "tech", "year"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU", "imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_imported_emissions_techs_Linear = (
    df_total_imported_emissions_EU_linear
    .groupby(["spore", "tech"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU","imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "imported_emissions_tonnes_total": "total_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_self" : "domesctic_emissions (2026-2050)",
        "imported_emissions_tonnes_EU": "EU_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_non_EU" : "non_EU_imported_emissions_tonnes (2026-2050)"
        
        })
    )

# Dataframe with import result totals from Logistic model

# Total imported emissions in the EU every year for every tech
df_total_imported_emissions_EU_logistic = (
    annual_imported_emissions_location_logistic
    .groupby(["spore", "tech", "year"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU", "imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_imported_emissions_techs_logistic = (
    df_total_imported_emissions_EU_logistic
    .groupby(["spore", "tech"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU","imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "imported_emissions_tonnes_total": "total_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_self" : "domesctic_emissions (2026-2050)",
        "imported_emissions_tonnes_EU": "EU_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_non_EU" : "non_EU_imported_emissions_tonnes (2026-2050)"
        
        })
    )

# Dataframe with import result totals from Logistic model

# Total imported emissions in the EU every year for every tech
df_total_imported_emissions_EU_gompertz = (
    annual_imported_emissions_location_gompertz
    .groupby(["spore", "tech", "year"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU", "imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_imported_emissions_techs_gompertz = (
    df_total_imported_emissions_EU_gompertz
    .groupby(["spore", "tech"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU","imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "imported_emissions_tonnes_total": "total_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_self" : "domesctic_emissions (2026-2050)",
        "imported_emissions_tonnes_EU": "EU_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_non_EU" : "non_EU_imported_emissions_tonnes (2026-2050)"
        
        })
    )

In [21]:
# Dataframe with annual construction emissions per country

# Replace "onshore wind" & "offshore wind" with wind (like in final_upstream_emissions_per_euro dataframe)
df_annual_investment_gompertz = df_annual_investment_gompertz.copy()
df_annual_investment_logistics = df_annual_investment_logistics.copy()
df_annual_investment_linear = df_annual_investment_linear.copy()

df_annual_investment_gompertz["tech"] = (
    df_annual_investment_gompertz["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)
df_annual_investment_logistics["tech"] = (
    df_annual_investment_logistics["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)
df_annual_investment_linear["tech"] = (
    df_annual_investment_linear["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)

# Linear model
annual_construction_emissions_per_country_linear = (
    df_annual_investment_linear
    .merge(
        df_final_upstream_emissions_per_euro_F.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)

annual_construction_emissions_per_country_linear["construction_cost"] = (
    annual_construction_emissions_per_country_linear["construction_cost"].fillna(0)
)

annual_construction_emissions_per_country_linear["construction_emissions_tonnes"] = (
    (annual_construction_emissions_per_country_linear["construction_cost"] *
    annual_construction_emissions_per_country_linear["total_upstream_emissions_per_sector"])
    /1000
)

annual_construction_emissions_per_country_linear = annual_construction_emissions_per_country_linear[
    [
        "spore",
        "country",
        "tech",
        "year",
        "construction_cost",
        "total_upstream_emissions_per_sector",
        "construction_emissions_tonnes"
    ]
].copy()

# Logistic model
annual_construction_emissions_per_country_logistic = (
    df_annual_investment_logistics
    .merge(
        df_final_upstream_emissions_per_euro_F.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)

annual_construction_emissions_per_country_logistic["construction_cost"] = (
    annual_construction_emissions_per_country_logistic["construction_cost"].fillna(0)
)

annual_construction_emissions_per_country_logistic["construction_emissions_tonnes"] = (
    (annual_construction_emissions_per_country_logistic["construction_cost"] *
    annual_construction_emissions_per_country_logistic["total_upstream_emissions_per_sector"])
    /1000
)

annual_construction_emissions_per_country_logistic = annual_construction_emissions_per_country_logistic[
    [
        "spore",
        "country",
        "tech",
        "year",
        "construction_cost",
        "total_upstream_emissions_per_sector",
        "construction_emissions_tonnes"
    ]
].copy()

# Gompertz model
annual_construction_emissions_per_country_gompertz = (
    df_annual_investment_gompertz
    .merge(
        df_final_upstream_emissions_per_euro_F.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)

annual_construction_emissions_per_country_gompertz["construction_cost"] = (
    annual_construction_emissions_per_country_gompertz["construction_cost"].fillna(0)
)

annual_construction_emissions_per_country_gompertz["construction_emissions_tonnes"] = (
    (annual_construction_emissions_per_country_gompertz["construction_cost"] *
    annual_construction_emissions_per_country_gompertz["total_upstream_emissions_per_sector"])
    /1000
)

annual_construction_emissions_per_country_gompertz = annual_construction_emissions_per_country_gompertz[
    [
        "spore",
        "country",
        "tech",
        "year",
        "construction_cost",
        "total_upstream_emissions_per_sector",
        "construction_emissions_tonnes"
    ]
].copy()

In [22]:
# Determine where construction emissions come from (domestic, EU, non-EU)

# Linear model
annual_construction_emissions_location_linear = (
    annual_construction_emissions_per_country_linear
    .merge(
        df_total_upstream_location_shares_F,
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)


annual_construction_emissions_location_linear["construction_emissions_tonnes_self"] = (
    annual_construction_emissions_location_linear["construction_emissions_tonnes"] *
    annual_construction_emissions_location_linear["share_self"]
)

annual_construction_emissions_location_linear["construction_emissions_tonnes_EU"] = (
    annual_construction_emissions_location_linear["construction_emissions_tonnes"] *
    annual_construction_emissions_location_linear["share_EU"]
)

annual_construction_emissions_location_linear["construction_emissions_tonnes_non_EU"] = (
    annual_construction_emissions_location_linear["construction_emissions_tonnes"] *
    annual_construction_emissions_location_linear["share_non_EU"]
)

annual_construction_emissions_location_linear["construction_emissions_tonnes_total"] = (
    annual_construction_emissions_location_linear["construction_emissions_tonnes"]
)


annual_construction_emissions_location_linear = annual_construction_emissions_location_linear[
    [
        "spore",
        "country",
        "tech",
        "year",
        "construction_emissions_tonnes_self",
        "construction_emissions_tonnes_EU",
        "construction_emissions_tonnes_non_EU",
        "construction_emissions_tonnes_total"
    ]
].copy()

# Logistic model
annual_construction_emissions_location_logistic = (
    annual_construction_emissions_per_country_logistic
    .merge(
        df_total_upstream_location_shares_F,
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)


annual_construction_emissions_location_logistic["construction_emissions_tonnes_self"] = (
    annual_construction_emissions_location_logistic["construction_emissions_tonnes"] *
    annual_construction_emissions_location_logistic["share_self"]
)

annual_construction_emissions_location_logistic["construction_emissions_tonnes_EU"] = (
    annual_construction_emissions_location_logistic["construction_emissions_tonnes"] *
    annual_construction_emissions_location_logistic["share_EU"]
)

annual_construction_emissions_location_logistic["construction_emissions_tonnes_non_EU"] = (
    annual_construction_emissions_location_logistic["construction_emissions_tonnes"] *
    annual_construction_emissions_location_logistic["share_non_EU"]
)

annual_construction_emissions_location_logistic["construction_emissions_tonnes_total"] = (
    annual_construction_emissions_location_logistic["construction_emissions_tonnes"]
)


annual_construction_emissions_location_logistic = annual_construction_emissions_location_logistic[
    [
        "spore",
        "country",
        "tech",
        "year",
        "construction_emissions_tonnes_self",
        "construction_emissions_tonnes_EU",
        "construction_emissions_tonnes_non_EU",
        "construction_emissions_tonnes_total"
    ]
].copy()

# Gompertz model
annual_construction_emissions_location_gompertz = (
    annual_construction_emissions_per_country_gompertz
    .merge(
        df_total_upstream_location_shares_F,
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)


annual_construction_emissions_location_gompertz["construction_emissions_tonnes_self"] = (
    annual_construction_emissions_location_gompertz["construction_emissions_tonnes"] *
    annual_construction_emissions_location_gompertz["share_self"]
)

annual_construction_emissions_location_gompertz["construction_emissions_tonnes_EU"] = (
    annual_construction_emissions_location_gompertz["construction_emissions_tonnes"] *
    annual_construction_emissions_location_gompertz["share_EU"]
)

annual_construction_emissions_location_gompertz["construction_emissions_tonnes_non_EU"] = (
    annual_construction_emissions_location_gompertz["construction_emissions_tonnes"] *
    annual_construction_emissions_location_gompertz["share_non_EU"]
)

annual_construction_emissions_location_gompertz["construction_emissions_tonnes_total"] = (
    annual_construction_emissions_location_gompertz["construction_emissions_tonnes"]
)



annual_construction_emissions_location_gompertz = annual_construction_emissions_location_gompertz[
    [
        "spore",
        "country",
        "tech",
        "year",
        "construction_emissions_tonnes_self",
        "construction_emissions_tonnes_EU",
        "construction_emissions_tonnes_non_EU",
        "construction_emissions_tonnes_total"
    ]
].copy()

In [23]:
# Dataframe with construction result totals

# Total imported emissions in the EU every year for every tech
df_total_construction_emissions_EU_linear = (
    annual_construction_emissions_location_linear
    .groupby(["spore", "tech", "year"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU", "construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_construction_emissions_techs = (
    df_total_construction_emissions_EU_linear
    .groupby(["spore", "tech"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU","construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "construction_emissions_tonnes_total": "total_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_self" : "domesctic_emissions_construction (2026-2050)",
        "construction_emissions_tonnes_EU": "EU_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_non_EU" : "non_EU_construction_emissions_tonnes (2026-2050)"
        
        })
    )

# Dataframe with construction result totals

# Total imported emissions in the EU every year for every tech
df_total_construction_emissions_EU_logistic = (
    annual_construction_emissions_location_logistic
    .groupby(["spore", "tech", "year"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU", "construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_construction_emissions_techs_logistic = (
    df_total_construction_emissions_EU_logistic
    .groupby(["spore", "tech"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU","construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "construction_emissions_tonnes_total": "total_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_self" : "domesctic_emissions_construction (2026-2050)",
        "construction_emissions_tonnes_EU": "EU_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_non_EU" : "non_EU_construction_emissions_tonnes (2026-2050)"
        
        })
    )
# Dataframe with construction result totals

# Total imported emissions in the EU every year for every tech
df_total_construction_emissions_EU_gompertz = (
    annual_construction_emissions_location_gompertz
    .groupby(["spore", "tech", "year"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU", "construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_construction_emissions_techs_gompertz = (
    df_total_construction_emissions_EU_gompertz
    .groupby(["spore", "tech"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU","construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "construction_emissions_tonnes_total": "total_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_self" : "domesctic_emissions_construction (2026-2050)",
        "construction_emissions_tonnes_EU": "EU_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_non_EU" : "non_EU_construction_emissions_tonnes (2026-2050)"
        
        })
    )

In [24]:
# Dataframe with total emissions (imported + construction)

annual_total_emissions_location_linear = (
annual_construction_emissions_location_linear
.merge(
    annual_imported_emissions_location_linear,
        on=["spore", "country", "tech", "year"],
        how="left"))


annual_total_emissions_location_linear["total_self"]= annual_total_emissions_location_linear["construction_emissions_tonnes_self"] + annual_total_emissions_location_linear["imported_emissions_tonnes_self"]
annual_total_emissions_location_linear["total_EU"]= annual_total_emissions_location_linear["construction_emissions_tonnes_EU"] + annual_total_emissions_location_linear["imported_emissions_tonnes_EU"]
annual_total_emissions_location_linear["total_non_EU"]= annual_total_emissions_location_linear["construction_emissions_tonnes_non_EU"] + annual_total_emissions_location_linear["imported_emissions_tonnes_non_EU"]
annual_total_emissions_location_linear["total"]= annual_total_emissions_location_linear["construction_emissions_tonnes_total"] + annual_total_emissions_location_linear["imported_emissions_tonnes_total"]


annual_total_emissions_location_linear = annual_total_emissions_location_linear[
    [
        "spore",
        "country",
        "tech",
        "year",
        "total_self",
        "total_EU",
        "total_non_EU",
        "total"
    ]
].copy()

annual_total_emissions_location_logistic = (
annual_construction_emissions_location_logistic
.merge(
    annual_imported_emissions_location_logistic,
        on=["spore", "country", "tech", "year"],
        how="left"))


annual_total_emissions_location_logistic["total_self"]= annual_total_emissions_location_logistic["construction_emissions_tonnes_self"] + annual_total_emissions_location_logistic["imported_emissions_tonnes_self"]
annual_total_emissions_location_logistic["total_EU"]= annual_total_emissions_location_logistic["construction_emissions_tonnes_EU"] + annual_total_emissions_location_logistic["imported_emissions_tonnes_EU"]
annual_total_emissions_location_logistic["total_non_EU"]= annual_total_emissions_location_logistic["construction_emissions_tonnes_non_EU"] + annual_total_emissions_location_logistic["imported_emissions_tonnes_non_EU"]
annual_total_emissions_location_logistic["total"]= annual_total_emissions_location_logistic["construction_emissions_tonnes_total"] + annual_total_emissions_location_logistic["imported_emissions_tonnes_total"]


annual_total_emissions_location_logistic = annual_total_emissions_location_logistic[
    [
        "spore",
        "country",
        "tech",
        "year",
        "total_self",
        "total_EU",
        "total_non_EU",
        "total"
    ]
].copy()

annual_total_emissions_location_gompertz = (
annual_construction_emissions_location_gompertz
.merge(
    annual_imported_emissions_location_gompertz,
        on=["spore", "country", "tech", "year"],
        how="left"))


annual_total_emissions_location_gompertz["total_self"]= annual_total_emissions_location_gompertz["construction_emissions_tonnes_self"] + annual_total_emissions_location_gompertz["imported_emissions_tonnes_self"]
annual_total_emissions_location_gompertz["total_EU"]= annual_total_emissions_location_gompertz["construction_emissions_tonnes_EU"] + annual_total_emissions_location_gompertz["imported_emissions_tonnes_EU"]
annual_total_emissions_location_gompertz["total_non_EU"]= annual_total_emissions_location_gompertz["construction_emissions_tonnes_non_EU"] + annual_total_emissions_location_gompertz["imported_emissions_tonnes_non_EU"]
annual_total_emissions_location_gompertz["total"]= annual_total_emissions_location_gompertz["construction_emissions_tonnes_total"] + annual_total_emissions_location_gompertz["imported_emissions_tonnes_total"]


annual_total_emissions_location_gompertz = annual_total_emissions_location_gompertz[
    [
        "spore",
        "country",
        "tech",
        "year",
        "total_self",
        "total_EU",
        "total_non_EU",
        "total"
    ]
].copy()

In [25]:
# Dataframe with EU totals

total_EU_Emissions_locations_linear = (
annual_total_emissions_location_linear
.groupby(["spore", "tech", "year"], as_index=False)[["total_self","total_EU","total_non_EU"]]
.sum()
    )

total_EU_Emissions_locations_logistic = (
annual_total_emissions_location_logistic
.groupby(["spore", "tech", "year"], as_index=False)[["total_self","total_EU","total_non_EU"]]
.sum()
    )

total_EU_Emissions_locations_gompertz = (
annual_total_emissions_location_gompertz
.groupby(["spore", "tech", "year"], as_index=False)[["total_self","total_EU","total_non_EU"]]
.sum()
    )

In [30]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set region scope
EU_CONSUMING_COUNTRIES = set(EU_countries)
EU_ORIGIN_COUNTRIES = set(EU_countries) | {"MLT"}

OUTPUT_DIR = Path("outputs/upstream_origin_tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Growth model input dataframes
growth_model_investments = {
    "linear": df_annual_investment_linear,
    "logistic": df_annual_investment_logistics,
    "gompertz": df_annual_investment_gompertz,
}


# Use full country and sector names (not codes)
# Sector names from GHG file, if available
if {"nace_r2_code", "nace_r2"}.issubset(df_GHG.columns):
    sector_name_map = (
        df_GHG[["nace_r2_code", "nace_r2"]]
        .dropna()
        .drop_duplicates(subset=["nace_r2_code"])
        .set_index("nace_r2_code")["nace_r2"]
        .to_dict()
    )
else:
    sector_name_map = {}

# Region names from regions file, if a readable column exists
possible_region_name_cols = [
    col for col in df_regions.columns
    if col.lower() in ["name", "country", "country_name", "region", "region_name"]
]

if "ISO_3" in df_regions.columns and len(possible_region_name_cols) > 0:
    region_name_col = possible_region_name_cols[0]
    region_name_map = (
        df_regions[["ISO_3", region_name_col]]
        .dropna()
        .drop_duplicates(subset=["ISO_3"])
        .set_index("ISO_3")[region_name_col]
        .to_dict()
    )
else:
    region_name_map = {}


# Determine Leontief inverse for non-EU 

def prepare_non_eu_upstream_coefficients():
    

    coeff = df_upstream_emissions_per_euro.copy()

    coeff["upstream_emissions_per_euro"] = (
        coeff["upstream_emissions_per_euro"]
        .fillna(0)
    )

    # Keep only emissions that occur outside the EU
    coeff = coeff.loc[
        ~coeff["country_orig"].isin(EU_ORIGIN_COUNTRIES)
    ].copy()

    # Only keep the sectors relevant for this analysis:
    # C27 = solar and batteries
    # C28 = wind
    # F   = construction
    relevant_destination_sectors = set(res_code.values())

    coeff = coeff.loc[
        coeff["sector_dest"].isin(relevant_destination_sectors)
    ].copy()

    return coeff


df_non_eu_upstream_coeff = prepare_non_eu_upstream_coefficients()




def prepare_investment_df(df_investment):
 
    df = df_investment.copy()

    df["tech"] = df["tech"].replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })

    df = df.loc[
        df["country"].isin(EU_CONSUMING_COUNTRIES)
    ].copy()

    df["investment"] = df["investment"].fillna(0)
    df["construction_cost"] = df["construction_cost"].fillna(0)

    return df


# ------------------------------------------------------------
# Calculate non-EU upstream emissions by origin region and sector
# ------------------------------------------------------------

def calculate_non_eu_origin_contributions(df_investment, model_name):
    """
    Calculates the origin of non-EU upstream emissions for one growth model.

    It includes:
    1. equipment/component investment emissions, allocated through import shares;
    2. construction emissions, allocated to sector F in the consuming EU country.

    Both parts use the Leontief-based upstream emissions per euro and only
    count emissions occurring outside the EU.
    """

    df_inv = prepare_investment_df(df_investment)

    # --------------------------------------------------------
    # 1. Equipment / imported technology component demand
    # --------------------------------------------------------

    equipment_demand = (
        df_inv
        .groupby(["spore", "country", "tech", "year"], as_index=False)["investment"]
        .sum()
    )

    equipment_demand["sector_dest"] = equipment_demand["tech"].map(res_code)

    # Add import shares:
    # country = EU consuming country
    # country_orig in df_import_shares = direct import partner / producer country
    equipment_demand = equipment_demand.merge(
        df_import_shares[
            ["country_dest", "country_orig", "tech", "import_share"]
        ],
        left_on=["country", "tech"],
        right_on=["country_dest", "tech"],
        how="left",
        validate="many_to_many"
    )

    # Diagnostics for missing import shares
    missing_import_shares = equipment_demand.loc[
        equipment_demand["import_share"].isna(),
        ["spore", "country", "tech"]
    ].drop_duplicates()

    if len(missing_import_shares) > 0:
        print(f"\nWarning: missing import shares in {model_name}:")
        print(missing_import_shares)

    equipment_demand["import_share"] = equipment_demand["import_share"].fillna(0)

    equipment_demand["weighted_equipment_demand_eur"] = (
        equipment_demand["investment"] *
        equipment_demand["import_share"]
    )

    equipment_demand = equipment_demand.rename(columns={
        "country_orig": "producer_country"
    })

    # Aggregate before merging with Leontief coefficients to avoid unnecessary rows
    equipment_demand_agg = (
        equipment_demand
        .groupby(["spore", "producer_country", "sector_dest"], as_index=False)
        ["weighted_equipment_demand_eur"]
        .sum()
        .rename(columns={
            "producer_country": "country_dest",
            "weighted_equipment_demand_eur": "final_demand_eur"
        })
    )

    equipment_demand_agg["demand_type"] = "equipment"

   

    # construction_demand_agg = (
    #     df_inv
    #     .groupby(["country"], as_index=False)["construction_cost"]
    #     .sum()
    #     .rename(columns={
    #         "country": "country_dest",
    #         "construction_cost": "final_demand_eur"
    #     })
    # )

    # construction_demand_agg["sector_dest"] = res_code["construction"]
    # construction_demand_agg["demand_type"] = "construction"

 

    total_demand = equipment_demand_agg
    # pd.concat(
    #     [
    #         equipment_demand_agg[
    #             ["country_dest", "sector_dest", "final_demand_eur", "demand_type"]
    #         ],
    #         construction_demand_agg[
    #             ["country_dest", "sector_dest", "final_demand_eur", "demand_type"]
    #         ]
    #     ],
    #     ignore_index=True
    # )

    total_demand = total_demand.loc[
        total_demand["final_demand_eur"] > 0
    ].copy()

    contributions = total_demand.merge(
        df_non_eu_upstream_coeff[
            [
                "country_dest",
                "sector_dest",
                "country_orig",
                "sector_orig",
                "upstream_emissions_per_euro"
            ]
        ],
        on=["country_dest", "sector_dest"],
        how="left",
        validate="many_to_many"
    )

    contributions["upstream_emissions_per_euro"] = (
        contributions["upstream_emissions_per_euro"].fillna(0)
    )

   
    # final demand [EUR] * emissions per euro / 1000 = tonnes CO2-eq
    contributions["non_eu_upstream_emissions_tonnes"] = (
        contributions["final_demand_eur"] *
        contributions["upstream_emissions_per_euro"]
        / 1000
    )

    contributions = contributions.loc[
        contributions["non_eu_upstream_emissions_tonnes"] > 0
    ].copy()

    total_non_eu_emissions_per_spore = (
        contributions
        .groupby("spore")["non_eu_upstream_emissions_tonnes"]
        .sum()
    )


    # Region shares

    region_table = (
    contributions
    .groupby(["spore", "country_orig"], as_index=False)["non_eu_upstream_emissions_tonnes"]
    .sum()
    .rename(columns={
        "country_orig": "region",
        "non_eu_upstream_emissions_tonnes": "emissions_tonnes"
    })
    )

    region_table["total_spore_emissions_tonnes"] = (
        region_table["spore"].map(total_non_eu_emissions_per_spore)
    )

    region_table["emissions_MtCO2eq"] = (
        region_table["emissions_tonnes"] / 1e6
    )

    region_table["percentage"] = np.where(
        region_table["total_spore_emissions_tonnes"] > 0,
        region_table["emissions_tonnes"] / region_table["total_spore_emissions_tonnes"] * 100,
        0
    )

    region_table["region_name"] = (
        region_table["region"]
        .map(region_name_map)
        .fillna(region_table["region"])
    )

    region_table = region_table[
        ["spore", "region", "region_name", "emissions_MtCO2eq", "percentage"]
    ].sort_values(["spore", "percentage"], ascending=[True, False])
    
   # Industriy shares

    industry_table = (
    contributions
    .groupby(["spore", "sector_orig"], as_index=False)["non_eu_upstream_emissions_tonnes"]
    .sum()
    .rename(columns={
        "sector_orig": "industry",
        "non_eu_upstream_emissions_tonnes": "emissions_tonnes"
    })
    )

    industry_table["total_spore_emissions_tonnes"] = (
        industry_table["spore"].map(total_non_eu_emissions_per_spore)
    )

    industry_table["emissions_MtCO2eq"] = (
        industry_table["emissions_tonnes"] / 1e6
    )

    industry_table["percentage"] = np.where(
        industry_table["total_spore_emissions_tonnes"] > 0,
        industry_table["emissions_tonnes"] / industry_table["total_spore_emissions_tonnes"] * 100,
        0
    )

    industry_table["industry_name"] = (
        industry_table["industry"]
        .map(sector_name_map)
        .fillna(industry_table["industry"])
    )

    industry_table = industry_table[
        ["spore", "industry", "industry_name", "emissions_MtCO2eq", "percentage"]
    ].sort_values(["spore", "percentage"], ascending=[True, False])


    return region_table, industry_table, contributions


# Loop over growth models

non_eu_region_tables = {}
non_eu_industry_tables = {}
non_eu_contribution_details = {}

for model_name, df_model in growth_model_investments.items():

    region_table, industry_table, contributions = calculate_non_eu_origin_contributions(
        df_model,
        model_name
    )

    non_eu_region_tables[model_name] = region_table
    non_eu_industry_tables[model_name] = industry_table
    non_eu_contribution_details[model_name] = contributions

    print(f"\n{model_name.upper()} - top 10 non-EU origin regions")
    display(region_table.head(10))

    print(f"\n{model_name.upper()} - top 10 non-EU origin industries")
    display(industry_table.head(10))


         spore country       tech
20050        0     FRA  batteries
20075        0     FRA      solar
20100        0     FRA       wind
70100        1     FRA  batteries
70125        1     FRA      solar
...        ...     ...        ...
3573625     71     FRA      solar
3573650     71     FRA       wind
3623650     72     FRA  batteries
3623675     72     FRA      solar
3623700     72     FRA       wind

[219 rows x 3 columns]

LINEAR - top 10 non-EU origin regions


,spore,region,region_name,emissions_MtCO2eq,percentage
13,0,ROW,ROW,28.614732,55.011465
5,0,CHN,CHN,11.000738,21.148781
14,0,RUS,RUS,3.221603,6.193492
16,0,TUR,TUR,1.248790,2.400784
18,0,ZAF,ZAF,1.189421,2.286647
8,0,IND,IND,0.996318,1.915409
6,0,GBR,GBR,0.897225,1.724903
17,0,USA,USA,0.722527,1.389048
9,0,JPN,JPN,0.713910,1.372483
15,0,SAU,SAU,0.654319,1.257920



LINEAR - top 10 non-EU origin industries


,spore,industry,industry_name,emissions_MtCO2eq,percentage
14,0,C24,Manufacture of basic metals,15.186766,29.196369
3,0,B,Mining and quarrying,12.658054,24.334951
23,0,D35,"Electricity, gas, steam and air conditioning s...",10.683693,20.539267
10,0,C20,Manufacture of chemicals and chemical products,3.146280,6.048685
25,0,E37T39,"Sewerage, waste management, remediation activi...",1.742886,3.350677
17,0,C27,Manufacture of electrical equipment,1.648898,3.169985
13,0,C23,Manufacture of other non-metallic mineral prod...,1.493800,2.871811
31,0,H50,Water transport,1.141251,2.194041
18,0,C28,Manufacture of machinery and equipment n.e.c.,0.912836,1.754917
30,0,H49,Land transport and transport via pipelines,0.717443,1.379274



         spore country       tech
20050        0     FRA  batteries
20075        0     FRA      solar
20100        0     FRA       wind
70100        1     FRA  batteries
70125        1     FRA      solar
...        ...     ...        ...
3573625     71     FRA      solar
3573650     71     FRA       wind
3623650     72     FRA  batteries
3623675     72     FRA      solar
3623700     72     FRA       wind

[219 rows x 3 columns]

LOGISTIC - top 10 non-EU origin regions


...

,spore,industry,industry_name,emissions_MtCO2eq,percentage
14,0,C24,Manufacture of basic metals,14.841430,29.196113
3,0,B,Mining and quarrying,12.370251,24.334801
23,0,D35,"Electricity, gas, steam and air conditioning s...",10.440843,20.539264
10,0,C20,Manufacture of chemicals and chemical products,3.074906,6.048967
25,0,E37T39,"Sewerage, waste management, remediation activi...",1.703214,3.350568
17,0,C27,Manufacture of electrical equipment,1.611827,3.170792
13,0,C23,Manufacture of other non-metallic mineral prod...,1.459909,2.871938
31,0,H50,Water transport,1.115283,2.193988
18,0,C28,Manufacture of machinery and equipment n.e.c.,0.891791,1.754334
30,0,H49,Land transport and transport via pipelines,0.701125,1.379255



         spore country       tech
20050        0     FRA  batteries
20075        0     FRA      solar
20100        0     FRA       wind
70100        1     FRA  batteries
70125        1     FRA      solar
...        ...     ...        ...
3573625     71     FRA      solar
3573650     71     FRA       wind
3623650     72     FRA  batteries
3623675     72     FRA      solar
3623700     72     FRA       wind

[219 rows x 3 columns]

GOMPERTZ - top 10 non-EU origin regions


,spore,region,region_name,emissions_MtCO2eq,percentage
13,0,ROW,ROW,29.382940,55.005929
5,0,CHN,CHN,11.304235,21.161938
14,0,RUS,RUS,3.307767,6.192260
16,0,TUR,TUR,1.281840,2.399650
18,0,ZAF,ZAF,1.221132,2.286003
8,0,IND,IND,1.022474,1.914108
6,0,GBR,GBR,0.920602,1.723401
17,0,USA,USA,0.741800,1.388677
9,0,JPN,JPN,0.732952,1.372112
15,0,SAU,SAU,0.671960,1.257934



GOMPERTZ - top 10 non-EU origin industries


,spore,industry,industry_name,emissions_MtCO2eq,percentage
14,0,C24,Manufacture of basic metals,15.593899,29.192344
3,0,B,Mining and quarrying,12.998596,24.333843
23,0,D35,"Electricity, gas, steam and air conditioning s...",10.972250,20.540450
10,0,C20,Manufacture of chemicals and chemical products,3.232880,6.052068
25,0,E37T39,"Sewerage, waste management, remediation activi...",1.789159,3.349371
17,0,C27,Manufacture of electrical equipment,1.696869,3.176600
13,0,C23,Manufacture of other non-metallic mineral prod...,1.534637,2.872897
31,0,H50,Water transport,1.171728,2.193516
18,0,C28,Manufacture of machinery and equipment n.e.c.,0.935092,1.750527
30,0,H49,Land transport and transport via pipelines,0.736654,1.379043


In [31]:
region_table_mean = (
    region_table
    .groupby(["region", "region_name"], as_index=False)["percentage"]
    .mean()
    .sort_values("percentage", ascending=False)
)

industry_table_mean = (
    industry_table
    .groupby(["industry", "industry_name"], as_index=False)["percentage"]
    .mean()
    .sort_values("percentage", ascending=False)
)